In [1]:
import numpy as np
import hubbard

NELEC = (1, 0)
NSITE = 2
T     = 1
U     = 4

h1, eri, mol, mf = hubbard.hubbard_mf(
    NSITE, T, U, NELEC, dm0=None, pbc=True, verbose=4)

# mo_a, mo_b = mf.mo_coeff[0][:, :NELEC[0]], mf.mo_coeff[1][:, :NELEC[1]]

mo_a = np.array([[1.0], [0.0]]) # electron pinned to site 0
mo_b = np.zeros((NSITE, 0))

print(mo_a)
print(mo_b)

qmc = hubbard.afqmc_setup(
    h1, U, NELEC, mo_a, mo_b, NSITE)

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      7.0.0-30-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu


******** 1 electron system ********

WARN: HOMO -1 >= LUMO -1

converged SCF energy = -1  <S^2> = 0.75  2S+1 = 2
[[1.]
 [0.]]
[]


In [2]:
import sys
import time
from dataclasses import dataclass, field
from functools import partial
from typing import Any, Optional, Sequence

import numpy as np

def setup_jax() -> None:
    """Configure jax for AFQMC (float64, CPU/GPU backend).  Safe to call twice."""
    import jax

    if not jax.config.jax_enable_x64:
        jax.config.update("jax_enable_x64", True)


setup_jax()

import jax.numpy as jnp                      # noqa: E402
from jax import jit, random                  # noqa: E402

from afqmc import fp_sampling, linalg_utils, prep, propagation, sampling  # noqa: E402
from afqmc.wavefunctions import wavefunctions_unrestricted as uwf         # noqa: E402

class propagator_uhf_gen(propagation.propagator_unrestricted):
    """
    Unrestricted phaseless propagator, plus a free-projection step that also works
    when one spin channel is empty.

    `afqmc`'s `propagate_free` divides by 2*nelec[sigma] when re-attaching the
    accumulated phase, which raises ZeroDivisionError for a fully polarised system.
    Multiplying every occupied column of both determinants by c gives a determinant
    factor c^(n_up) * c^(n_dn) = c^(N_e), so c = phase**(1/N_e) works in general.
    (It also fixes `exp(dt*h0_prop + e_estimate)` -> `exp(dt*(h0_prop + e_estimate))`;
    that factor is walker independent and cancels in the estimator, but it can
    overflow.)
    """

    @partial(jit, static_argnums=(0, 1))
    def propagate_free(self, trial, ham_data, prop_data, fields):
        n_tot = trial.nelec[0] + trial.nelec[1]

        shift_term = jnp.einsum("wg,g->w", fields, ham_data["mf_shifts"])
        constants = jnp.exp(-jnp.sqrt(self.dt) * shift_term) * jnp.exp(
            self.dt * (ham_data["h0_prop"] + prop_data["e_estimate"]))

        constants_abs = jnp.abs(constants)
        phase = constants / constants_abs

        prop_data["weights"] = prop_data["weights"] * constants_abs
        prop_data["walkers"] = self._apply_trotprop(ham_data, prop_data["walkers"], fields)

        c = phase ** (1.0 / n_tot)
        prop_data["walkers"] = self._multiply_constant(prop_data["walkers"], jnp.stack((c, c)))
        return prop_data
    
@dataclass
class HubbardSystem:
    """Everything the drivers need: Hamiltonian, trial, and their intermediates."""
    ham: Any
    ham_data: dict
    nchol: int
    trial: Any
    wave_data: dict
    h1: np.ndarray
    u: float
    nelec: tuple
    norb: int

    @property
    def trial_vector(self) -> np.ndarray:
        """Occupied alpha trial orbitals as a real numpy array (norb, nocc)."""
        return np.asarray(self.wave_data["mo_coeff"][0]).real

@dataclass
class QMCResult:
    """Block energies plus whatever walker snapshots were requested."""
    method: str
    tau: np.ndarray                       # (n_blocks,)
    energy: np.ndarray                    # (n_blocks,) averaged over runs
    error: np.ndarray                     # (n_blocks,) scatter between runs
    block_energies: np.ndarray            # (n_runs, n_blocks) raw
    block_weights: Optional[np.ndarray]   # (n_runs, n_blocks), free projection only
    snaps: list = field(default_factory=list)
    log_amplitude: Optional[np.ndarray] = None   # (n_traj, n_samples), free projection
    ess: Optional[np.ndarray] = None             # effective no. of trajectories vs tau
    tau_eq: float = 0.0
    n_killed: int = 0
    walltime: float = 0.0
    prop_data: Any = None

    # -- convenience ------------------------------------------------------------
    @property
    def n_runs(self) -> int:
        return self.block_energies.shape[0]

    def snap_taus(self) -> np.ndarray:
        return np.array(sorted({round(s.tau, 8) for s in self.snaps}))

    def snaps_at(self, tau: float, atol: float = 1e-6) -> list:
        return [s for s in self.snaps if abs(s.tau - tau) < atol]

    def plateau(self, tau_eq: Optional[float] = None, n_seg: int = 1):
        """
        Converged energy: mean and error from the scatter of the independent runs.

        Block energies inside one run are strongly autocorrelated, so the naive
        per-block standard error is several times too small; averaging whole runs
        (optionally split into `n_seg` long contiguous segments) avoids that.
        """
        tau_eq = self.tau_eq if tau_eq is None else tau_eq
        mask = self.tau > tau_eq
        if not mask.any():
            raise ValueError("no blocks with tau > %g" % tau_eq)
        segs = np.array_split(self.block_energies[:, mask], n_seg, axis=1)
        means = np.concatenate([s.mean(axis=1) for s in segs])
        err = means.std(ddof=1) / np.sqrt(means.size) if means.size > 1 else np.nan
        return float(means.mean()), float(err)


In [3]:
class propagator_uhf_gen(propagation.propagator_unrestricted):
    """
    Unrestricted phaseless propagator, plus a free-projection step that also works
    when one spin channel is empty.

    `afqmc`'s `propagate_free` divides by 2*nelec[sigma] when re-attaching the
    accumulated phase, which raises ZeroDivisionError for a fully polarised system.
    Multiplying every occupied column of both determinants by c gives a determinant
    factor c^(n_up) * c^(n_dn) = c^(N_e), so c = phase**(1/N_e) works in general.
    (It also fixes `exp(dt*h0_prop + e_estimate)` -> `exp(dt*(h0_prop + e_estimate))`;
    that factor is walker independent and cancels in the estimator, but it can
    overflow.)
    """

    @partial(jit, static_argnums=(0, 1))
    def propagate_free(self, trial, ham_data, prop_data, fields):
        n_tot = trial.nelec[0] + trial.nelec[1]

        shift_term = jnp.einsum("wg,g->w", fields, ham_data["mf_shifts"])
        constants = jnp.exp(-jnp.sqrt(self.dt) * shift_term) * jnp.exp(
            self.dt * (ham_data["h0_prop"] + prop_data["e_estimate"]))

        constants_abs = jnp.abs(constants)
        phase = constants / constants_abs

        prop_data["weights"] = prop_data["weights"] * constants_abs
        prop_data["walkers"] = self._apply_trotprop(ham_data, prop_data["walkers"], fields)

        c = phase ** (1.0 / n_tot)
        prop_data["walkers"] = self._multiply_constant(prop_data["walkers"], jnp.stack((c, c)))
        return prop_data


@dataclass
class fp_sampler_gen(fp_sampling.fp_sampler):
    """
    `fp_sampler` with two changes.

    1.  The N_e-column phase redistribution, so a fully polarised system works.

    2.  The per-step renormalisation `W <- n_walkers * W / sum(W)` is *recorded*
        rather than discarded.  That factor is the trajectory's accumulated
        amplitude: it cancels inside one block (the block energy is a ratio, and
        the factor is common to every walker), but it is exactly what weights one
        trajectory against another.  Dropping it biases every average over
        trajectories.  We keep the renormalisation -- without it the weights
        overflow -- and accumulate log(sum(W)/n_walkers) in `prop_data["log_norm"]`,
        so no information is lost.

    `sr_every` sets the stochastic-reconfiguration cadence in steps (1 = every
    step, as in `afqmc`; 0 = never).  SR conserves the total weight, so it does not
    interfere with the bookkeeping -- but reconfiguring every step forces every
    walker weight to the mean, which is strong population control and carries its
    own O(1/n_walkers) bias.
    """

    sr_every: int = 1

    def __hash__(self) -> int:
        # @dataclass regenerates __eq__ and drops __hash__; both must see sr_every
        # or jit will silently reuse a trace made for a different cadence.
        return hash(tuple(self.__dict__.values()))

    @partial(jit, static_argnums=(0, 4, 5))
    def _step_scan(self, prop_data, fields, ham_data, prop, trial, wave_data):
        prop_data = prop.propagate_free(trial, ham_data, prop_data, fields)

        prop_data["walkers"], norms = linalg_utils.qr_vmap_uhf(prop_data["walkers"])
        norms = norms[0] * norms[1]
        norms_abs = jnp.abs(norms)
        phase = norms / norms_abs

        n_walkers = int(prop_data["weights"].shape[0])
        prop_data["weights"] = prop_data["weights"] * norms_abs

        total = jnp.sum(prop_data["weights"])
        prop_data["log_norm"] = prop_data["log_norm"] + jnp.log(total / n_walkers)
        prop_data["weights"] = n_walkers * prop_data["weights"] / total

        n_tot = trial.nelec[0] + trial.nelec[1]
        c = phase ** (1.0 / n_tot)
        prop_data["walkers"] = prop._multiply_constant(prop_data["walkers"], jnp.stack((c, c)))

        prop_data["step"] = prop_data["step"] + 1
        if self.sr_every:
            do_sr = (prop_data["step"] % self.sr_every) == 0
            keep_w = [prop_data["walkers"][0], prop_data["walkers"][1]]
            keep_wt = prop_data["weights"]
            prop_data = prop.stochastic_reconfiguration_local(prop_data)
            prop_data["walkers"] = [jnp.where(do_sr, prop_data["walkers"][k], keep_w[k])
                                    for k in (0, 1)]
            prop_data["weights"] = jnp.where(do_sr, prop_data["weights"], keep_wt)
        return prop_data, fields

In [4]:
def _fp_estimate(log_amp, es, sign=None):
    """
    Free-projection mixed estimator, amplitude-weighted over trajectories.

        E(tau) = sum_i A_i(tau) e_i(tau) / sum_i A_i(tau)

    where A_i is trajectory i's unnormalised <Psi_T|exp(-tau H)|Phi_0> and e_i its
    block energy.  A_i spans many orders of magnitude, so it is carried as
    log|A_i| (+ a sign) and the ratio is formed after subtracting the largest
    exponent.

    log_amp, es, sign : (n_samples, n_traj).  Trajectories are the SECOND axis, so
    a running estimate over the first i+1 of them slices as `log_amp[:, :i+1]`.

    Returns (relative amplitude, energy, error, effective no. of trajectories).
    The error is a jackknife over trajectories: for a ratio of sums with unequal
    weights, `std(e)/sqrt(n)` is not the right error.
    """
    log_amp = np.atleast_2d(np.real(log_amp))
    es = np.atleast_2d(np.real(es))
    n_traj = log_amp.shape[1]
    sign = np.ones_like(log_amp) if sign is None else np.atleast_2d(np.real(sign))

    a = sign * np.exp(log_amp - log_amp.max(axis=1, keepdims=True))    # (n_samples, n_traj)
    num, den = (a * es).sum(1), a.sum(1)
    mean = num / den

    # amplitude relative to tau = 0, for display: how fast the projection decays
    amp = np.exp(log_amp - log_amp[0].max()).mean(axis=1)

    if n_traj > 1:
        jk = (num[:, None] - a * es) / (den[:, None] - a)              # leave one out
        err = np.sqrt((n_traj - 1) / n_traj
                      * ((jk - jk.mean(axis=1, keepdims=True)) ** 2).sum(axis=1))
    else:
        err = np.full(mean.shape, np.nan)

    ess = np.abs(a).sum(1) ** 2 / (a ** 2).sum(1)                      # Kish
    return amp, mean, err, ess

In [5]:
def run_free_projection(system: HubbardSystem, *,
                        dt=0.0025, n_walkers=500, n_prop_steps=40, n_blocks=30,
                        n_traj=16, seed=0, n_exp_terms=10, sr_every=1,
                        snap_every=0, verbose=0) -> QMCResult:
    """
    Free-projection AFQMC: no force bias, no constraint, exact but with a variance
    that grows with tau.  The estimator is the mixed estimator with the trial
    overlap kept explicitly in the weight, averaged over independent trajectories.

    The returned arrays have n_blocks + 1 entries: entry 0 is the trial at tau = 0,
    entry b + 1 is block b, following afqmc/scripts/run_fpafqmc.py.

    Trajectories are combined by their accumulated amplitude
    A_i = <Psi_T|exp(-tau H)|Phi_0>_i, carried in logs, not by their block weight
    alone -- see `fp_sampler_gen` and `_fp_estimate`.

    sr_every   : stochastic-reconfiguration cadence in propagation steps.  1 (the
                 default, and what `afqmc` does) resets every walker weight to the
                 mean at every step: strong population control, small error bars,
                 an O(1/n_walkers) bias.  Larger values, or 0 for never, give a
                 truer free projection at the cost of a collapsing effective
                 sample size -- watch the ESS column.
    snap_every : store a walker snapshot every this many blocks, in every
                 trajectory (0 = none).  Also sets which tau rows get printed.
                 No snapshot is taken at tau = 0, where every walker is the trial.
    verbose    : 0 silent; 1 the running E(tau) table (throttled to every
                 n_traj // 10 trajectories); 2 also the reconstructed |Psi> at each
                 snapshot (one electron on two sites only).
    """
    prop = propagator_uhf_gen(dt=dt, n_walkers=n_walkers,
                              n_exp_terms=n_exp_terms, n_batch=1)
    hd = system.ham.build_propagation_intermediates(
        dict(system.ham_data), prop, system.trial, system.wave_data)
    smp = fp_sampler_gen(n_prop_steps=n_prop_steps, n_eql_blocks=n_blocks,
                         n_trj=n_traj, n_chol=system.nchol, sr_every=sr_every)

    # Sample 0 is the trial at tau = 0, as in afqmc/scripts/run_fpafqmc.py, so every
    # sample array has n_blocks + 1 rows and row b + 1 holds block b.
    tau = dt * n_prop_steps * np.arange(0, n_blocks + 1)
    row_every = snap_every if snap_every else max(1, n_blocks // 10)
    rows = list(range(0, n_blocks + 1, row_every))
    if rows[-1] != n_blocks:
        rows.append(n_blocks)
    traj_every = max(1, n_traj // 10)          # as in afqmc/scripts/run_fpafqmc.py
    analyse = verbose >= 2 and hubbard._can_analyse(system)
    t0 = time.time()

    # log|A| and sign(A) of each trajectory's <Psi_T|exp(-tau H)|Phi_0>, plus the
    # raw block weights and energies.  (n_samples, n_traj) throughout.
    log_amp = np.zeros((n_blocks + 1, n_traj))
    sgn = np.ones((n_blocks + 1, n_traj))
    ws = np.zeros((n_blocks + 1, n_traj))
    es = np.zeros((n_blocks + 1, n_traj))
    snaps, pd = [], None

    for i, key in enumerate(random.split(random.PRNGKey(seed), n_traj)):
        pd = prop.init_prop_data(system.trial, system.wave_data, hd, None)
        pd["key"] = key
        pd["log_norm"] = jnp.array(0.0)        # accumulated renormalisation, in logs
        pd["step"] = jnp.array(0)

        w0 = float(np.real(jnp.sum(pd["weights"] * pd["overlaps"])))
        ws[0, i] = w0
        es[0, i] = float(np.real(pd["e_estimate"]))
        log_amp[0, i] = np.log(abs(w0))
        sgn[0, i] = np.sign(w0)

        # A Python loop over `fp_block` rather than `scan_eql_blocks`: bit-identical
        # results for ~4% more time, and it keeps the intermediate walkers reachable
        # so that snap_every can work.
        for b in range(n_blocks):
            pd, (blk_w, blk_e) = smp.fp_block(pd, hd, prop, system.trial,
                                              system.wave_data)
            w = float(np.real(complex(blk_w)))
            ws[b + 1, i] = w
            es[b + 1, i] = float(np.real(complex(blk_e)))
            log_amp[b + 1, i] = float(pd["log_norm"]) + np.log(abs(w))
            sgn[b + 1, i] = np.sign(w)
            if snap_every and (b + 1) % snap_every == 0:
                snaps.append(hubbard._snapshot(pd, i, tau[b + 1], importance=False))

        if verbose and (i == 0 or (i + 1) % traj_every == 0 or i == n_traj - 1):
            amp_i, mean_i, err_i, ess_i = _fp_estimate(
                log_amp[:, :i + 1], es[:, :i + 1], sgn[:, :i + 1])
            print("\nFree projection | trajectories 1-%d of %d | %.1f s"
                  % (i + 1, n_traj, time.time() - t0))
            print("  %6s  %10s  %10s  %9s  %6s"
                  % ("tau", "amplitude", "Energy", "Error", "ESS"))
            for b in rows:
                e_str = "%9s" % "N/A" if np.isnan(err_i[b]) else "%9.5f" % err_i[b]
                print("  %6.2f  %10.3e  %10.5f  %s  %6.1f"
                      % (tau[b], amp_i[b], mean_i[b], e_str, ess_i[b]))
                if analyse and snap_every:
                    # running estimate over every trajectory finished so far, so
                    # this line matches the energy column above it
                    hit = [sn for sn in snaps if abs(sn.tau - tau[b]) < 1e-9]
                    if hit:
                        print(hubbard._snap_line(hit, system))

    amp, mean, err, ess = _fp_estimate(log_amp, es, sgn)

    if verbose:
        print("\n" + "=" * 68)
        print("  Free projection | %d trajectories x %d blocks | sr_every = %d | %.1f s"
              % (n_traj, n_blocks, sr_every, time.time() - t0))
        print("  E(tau = %.2f) = %+.5f +/- %.5f" % (tau[-1], mean[-1], err[-1]))
        print("  amplitude relative to tau = 0 : %.3e | effective trajectories: %.1f / %d"
              % (amp[-1], ess[-1], n_traj))
        if ess[-1] < 0.5 * n_traj:
            print("  WARNING: the trajectory amplitudes have spread out; the free")
            print("  projection is losing statistical power at this tau.")
        if analyse and snaps:
            tail = [sn for sn in snaps if sn.tau >= tau[-1] - 1e-9]
            if tail:
                r = hubbard.wavefunction(tail, system.h1, system.trial_vector)
                print("  |Psi> at tau = %.2f : theta = %+.5f +/- %.5f, "
                      "c = (%+.5f, %+.5f)" % (tail[0].tau, r["theta"], r["theta_err"],
                                              r["c1"], r["c2"]))
        print("=" * 68)

    return QMCResult(method="free projection", tau=tau,
                     energy=mean, error=err,
                     block_energies=es.T,               # (n_traj, n_samples)
                     block_weights=ws.T,
                     log_amplitude=log_amp.T, ess=ess,
                     snaps=snaps, tau_eq=0.0, n_killed=0,
                     walltime=time.time() - t0, prop_data=pd)

In [6]:
qmc_result = run_free_projection(
                qmc, dt=0.005, n_walkers=500, 
                n_prop_steps=20, n_blocks=50,
                n_traj=50, seed=0, n_exp_terms=10, sr_every=10,
                snap_every=10, verbose=4
                )


Free projection | trajectories 1-1 of 50 | 4.0 s
     tau   amplitude      Energy      Error     ESS
    0.00   1.000e+00     0.00000        N/A     1.0
    1.00   1.532e+00    -0.69001        N/A     1.0
        |Psi> :  theta = +0.60399   c = (+0.82307, +0.56793)   E_var = -0.934903
    2.00   3.464e+00    -0.95508        N/A     1.0
        |Psi> :  theta = +0.76732   c = (+0.71977, +0.69421)   E_var = -0.999347
    3.00   9.271e+00    -0.89083        N/A     1.0
        |Psi> :  theta = +0.73108   c = (+0.74446, +0.66767)   E_var = -0.994104
    4.00   2.313e+01    -0.90679        N/A     1.0
        |Psi> :  theta = +0.73655   c = (+0.74079, +0.67174)   E_var = -0.995232
    5.00   5.823e+01    -0.81415        N/A     1.0
        |Psi> :  theta = +0.68331   c = (+0.77549, +0.63136)   E_var = -0.979227

Free projection | trajectories 1-5 of 50 | 6.2 s
     tau   amplitude      Energy      Error     ESS
    0.00   1.000e+00     0.00000    0.00000     5.0
    1.00   1.544e+00    -0.